In [18]:
import numpy as np
import pandas as pd

In [19]:
#load the Patient dataset
patient = pd.read_csv("/content/Patient_Data.csv")
patient

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [20]:
patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [21]:
#load the billing dataset
Billing = pd.read_csv("/content/Billing_Data.csv")
Billing

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [22]:
Billing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   PatientID         5 non-null      int64
 1   InsuranceCovered  5 non-null      int64
 2   FinalAmount       5 non-null      int64
dtypes: int64(3)
memory usage: 252.0 bytes


In [23]:
#Select only the columns relevant for billing
billing_relevant = patient[['PatientID', 'Department', 'Doctor', 'BillAmount']]
billing_relevant

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [24]:
#Drop administrative columns
patient.drop(['ReceptionistID', 'CheckInTime'], axis=1, inplace=True)
patient

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


In [25]:
#use groupby to find total bill amount per department
total_bill = patient.groupby('Department')['BillAmount'].sum().reset_index()
total_bill

,Department,BillAmount
0,Cardiology,16200.0
1,Dermatology,0.0
2,Neurology,0.0
3,Orthopedics,7500.0


In [26]:
#Remove duplicate patient records based on patientID
patient.drop_duplicates(subset='PatientID', inplace=True)
patient

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [27]:
#Fill missing BillAmount values with the mean bill amount
patient.fillna({'BillAmount': int(patient['BillAmount'].mean())}, inplace=True)
patient

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,6233.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,6233.0


In [28]:
#Merge the billing dataset with patient dataset on PatientID.
merge_Data = pd.merge(patient, Billing, on='PatientID', how='left')
merge_Data

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,2000,3000
1,102,Bob,Neurology,Dr. John,6233.0,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.0,1000,4000


In [29]:
#Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
new_patient = pd.DataFrame({
    'PatientID': [106, 107],
    'Name': ['Sham', 'Om'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Smith', 'Dr. Rose'],
    'BillAmount': [5000, 7000]
    })
patient = pd.concat([patient, new_patient], ignore_index=True)
patient

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,6233.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,6233.0
5,106,Sham,Cardiology,Dr. Smith,5000.0
6,107,Om,Neurology,Dr. Rose,7000.0


In [30]:
#Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).
extra_col = pd.DataFrame({
    'InsuranceCovered': ['Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No'][:len(patient)],
    'FinalAmount': patient['BillAmount'] * 0.8  # e.g., after 20% insurance cover
})
patient = pd.concat([patient, extra_col], axis=1)
patient


,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,Yes,4000.0
1,102,Bob,Neurology,Dr. John,6233.0,No,4986.4
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,Yes,6000.0
3,104,David,Cardiology,Dr. Smith,6200.0,Yes,4960.0
4,105,Eva,Dermatology,Dr. Rose,6233.0,No,4986.4
5,106,Sham,Cardiology,Dr. Smith,5000.0,Yes,4000.0
6,107,Om,Neurology,Dr. Rose,7000.0,No,5600.0
